# Train the legal rhetorical-role labeller on Google Colab

Connect to a hosted CPU and choose **Runtime → Run all**. This notebook trains two hierarchical BiLSTM-CRFs from scratch on the 50-case corpus. The validation fit learns from 32 judgments and is evaluated on eight. The final fit learns from all 40 training judgments and is evaluated on ten held-out judgments. Both fits use eight epochs and their own training-only vocabulary.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import json
import os
import platform
import subprocess
import sys
import tempfile

import google.colab
import sklearn
import torch

content = Path("/content")
if not content.is_dir():
    raise RuntimeError("Connect to a hosted Google Colab runtime.")
work = Path(tempfile.mkdtemp(prefix="legal-lexrank-", dir=content))
source = work / "source"
bundle = work / "role-study"
started = datetime.now(timezone.utc)
repository = "https://github.com/SrikarKorrapati/Extractive-Summarisation-of-Legal-Document.git"
revision = "dce96247aae788906f973faa97ae7bbfb953749f"
subprocess.run(["git", "init", "--quiet", str(source)], check=True)
subprocess.run(["git", "fetch", "--quiet", "--depth=1", repository, revision], cwd=source, check=True)
subprocess.run(["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"], cwd=source, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=source, text=True).strip() == revision
print(f"Training on hosted CPU with PyTorch {torch.__version__}, revision {revision[:12]}")

## Fit and evaluate

Each word-level bidirectional LSTM reads complete sentences. A second bidirectional LSTM and a linear-chain CRF predict the role sequence. The training command saves and reloads each checkpoint before scoring it. The validation and final checkpoints are kept separately for the summarization benchmark.

In [ ]:
subprocess.run(
    [sys.executable, "-u", "-m", "legal_lexrank.role_study",
     "--data", str(source), "--output", str(bundle), "--seed", "42", "--epochs", "8"],
    cwd=source, check=True,
)

## Save the Colab training run

Download **legal_lexrank_role_labeller.zip** from the Files pane after this cell finishes. It contains both checkpoints, their checksums, the case split, per-role scores and runtime details. Pass the extracted folder to the benchmark with `--role-run` to reuse these fits.

In [ ]:
metadata = {
    "platform": "Google Colab", "device": "cpu", "source_revision": revision,
    "release": os.environ.get("COLAB_RELEASE_TAG"), "os": platform.platform(),
    "python": platform.python_version(), "torch": str(torch.__version__), "sklearn": sklearn.__version__,
    "started_utc": started.isoformat(), "finished_utc": datetime.now(timezone.utc).isoformat(),
}
(bundle / "runtime.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
archive = content / "legal_lexrank_role_labeller.zip"
staging = work / archive.name
with ZipFile(staging, "w", compression=ZIP_DEFLATED) as output:
    for artifact in sorted(bundle.iterdir()):
        output.write(artifact, artifact.name)
staging.replace(archive)
reports = json.loads((bundle / "role_metrics.json").read_text(encoding="utf-8"))
for phase in ("validation", "test"):
    report = reports[phase]
    print(f"{phase.title()}: {report['accuracy']:.2%} accuracy, "
          f"{report['macro avg']['f1-score']:.4f} macro F1, "
          f"{int(report['macro avg']['support']):,} labelled sentences")
print(f"Training complete. Download {archive}")